# California Choropleth Maps

These maps use the merged county-level cholesterol and income data.

In [13]:
import geopandas as gpd
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display
from pathlib import Path

county = gpd.read_file("https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json")
ca_county = county[county["STATE"] == "06"].copy()
ca_county["countyfips"] = ca_county["id"].astype(str).str.zfill(5)
ca_geojson = ca_county.__geo_interface__


In [14]:
project_dir_options = [
    Path("26-the-backpropagators-analysis/cholestrol_income"),
    Path("cholestrol_income"),
    Path("."),
    Path(".."),
]

project_dir = next(
    path
    for path in project_dir_options
    if (path / "data/processed/cholesterol_income_by_county.csv").exists()
)

county_data = pd.read_csv(
    project_dir / "data/processed/cholesterol_income_by_county.csv",
    dtype={"countyfips": str},
)
county_data["countyfips"] = county_data["countyfips"].str.zfill(5)
county_data.head()


,stateabbr,countyname,countyfips,cholscreen_adjprev,income_year,per_capita_income,population
0,CA,Alameda,06001,86.5,2023,106657,1644199
1,CA,Alpine,06003,85.2,2023,75395,1166
2,CA,Amador,06005,84.0,2023,50020,40028
3,CA,Butte,06007,83.3,2023,56847,205741
4,CA,Calaveras,06009,83.6,2023,58425,44616


In [ ]:
fig_1_cholesterol_map = px.choropleth(
    county_data,
    geojson=ca_geojson,
    locations="countyfips",
    featureidkey="properties.countyfips",
    color="cholscreen_adjprev",
    color_continuous_scale="Blues",
    hover_name="countyname",
    hover_data={
        "countyfips": False,
        "cholscreen_adjprev": ":.1f",
        "per_capita_income": ":$,.0f",
        "population": ":,",
    },
    labels={"cholscreen_adjprev": "Cholesterol screening (%)"},
    
)

fig_1_cholesterol_map.update_geos(fitbounds="locations", visible=False)
fig_1_cholesterol_map.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig_1_cholesterol_map.show()


In [ ]:
fig_2_income_map = px.choropleth(
    county_data,
    geojson=ca_geojson,
    locations="countyfips",
    featureidkey="properties.countyfips",
    color="per_capita_income",
    color_continuous_scale="Greens",
    hover_name="countyname",
    hover_data={
        "countyfips": False,
        "per_capita_income": ":$,.0f",
        "cholscreen_adjprev": ":.1f",
        "population": ":,",
    },
    labels={"per_capita_income": "Per capita income ($)"},

)

fig_2_income_map.update_geos(fitbounds="locations", visible=False)
fig_2_income_map.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig_2_income_map.show()



# Scatter Plots

In [17]:
import numpy as np
import plotly.graph_objects as go

scatter_data = county_data.dropna(
    subset=["per_capita_income", "cholscreen_adjprev"]
).copy()

correlation = scatter_data["per_capita_income"].corr(
    scatter_data["cholscreen_adjprev"]
)

slope, intercept = np.polyfit(
    scatter_data["per_capita_income"],
    scatter_data["cholscreen_adjprev"],
    1,
)
trend_x = np.linspace(
    scatter_data["per_capita_income"].min(),
    scatter_data["per_capita_income"].max(),
    100,
)
trend_y = slope * trend_x + intercept

fig_4_income_cholesterol_scatter = px.scatter(
    scatter_data,
    x="per_capita_income",
    y="cholscreen_adjprev",
    hover_name="countyname",
    hover_data={
        "countyfips": False,
        "per_capita_income": ":$,.0f",
        "cholscreen_adjprev": ":.1f",
        "population": ":,",
    },
    labels={
        "per_capita_income": "Per capita income ($)",
        "cholscreen_adjprev": "Cholesterol screening (%)",
    },

)

fig_4_income_cholesterol_scatter.add_trace(
    go.Scatter(
        x=trend_x,
        y=trend_y,
        mode="lines",
        name="Trend line",
        line=dict(color="#d62728", width=2),
        hoverinfo="skip",
    )
)

fig_4_income_cholesterol_scatter.update_traces(
    marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color="white")),
    selector=dict(mode="markers"),
)
fig_4_income_cholesterol_scatter.update_layout(
    template="plotly_white",
    xaxis_tickprefix="$",
    xaxis_tickformat=",",
    yaxis_ticksuffix="%",
    margin=dict(l=60, r=30, t=70, b=60),
)

fig_4_income_cholesterol_scatter.write_html(
    project_dir / "results/figures/cholesterol_income_scatter.html"
)

print(f"Correlation: {correlation:.3f}")
fig_4_income_cholesterol_scatter.show()

Correlation: 0.785
